In [ ]:
import os, glob, zipfile
import numpy as np
import tensorflow as tf

# 1. Auto-find and extract any uploaded UCI HAR zip in /content/
for zpath in glob.glob("/content/*.zip"):
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall("/content/uci_extracted")

# Handle nested zip if downloaded directly from the new UCI portal
for nested in glob.glob("/content/uci_extracted/**/*.zip", recursive=True):
    with zipfile.ZipFile(nested, "r") as z:
        z.extractall("/content/uci_extracted")

# Locate the 'UCI HAR Dataset' directory
matches = glob.glob("/content/uci_extracted/**/UCI HAR Dataset", recursive=True)
assert matches, "Could not find 'UCI HAR Dataset'. Check that the .zip finished uploading."
DATA_DIR = matches[0]
print("Found dataset at:", DATA_DIR)

SIGNALS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

def load_split(split):
    signal_arrays = []
    for sig in SIGNALS:
        path = os.path.join(DATA_DIR, split, "Inertial Signals", f"{sig}_{split}.txt")
        signal_arrays.append(np.loadtxt(path, dtype=np.float32))
    # Shape: (samples, 128 timesteps, 9 channels)
    X = np.transpose(np.array(signal_arrays), (1, 2, 0))
    y_path = os.path.join(DATA_DIR, split, f"y_{split}.txt")
    y = np.loadtxt(y_path, dtype=np.int32) - 1  # Convert labels 1..6 -> 0..5
    return X, y

X_train, y_train = load_split("train")
X_test, y_test = load_split("test")

print(f"Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test:  {X_test.shape},  Labels: {y_test.shape}")
print("TensorFlow version:", tf.__version__)

Found dataset at: /content/uci_extracted/UCI HAR Dataset
Train: (7352, 128, 9), Labels: (7352,)
Test:  (2947, 128, 9),  Labels: (2947,)
TensorFlow version: 2.21.0


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Set seeds for repeatability
tf.keras.utils.set_random_seed(42)

MODEL_CONFIGS = {
    "small":  {"f1": 8,  "f2": 16, "dense": 16},
    "medium": {"f1": 16, "f2": 32, "dense": 32},
    "large":  {"f1": 32, "f2": 64, "dense": 64},
}

def build_har_cnn(f1, f2, dense_units, input_shape=(128, 9), num_classes=6):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(f1, kernel_size=5, padding="same", activation="relu"),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(f2, kernel_size=5, padding="same", activation="relu"),
        layers.MaxPooling1D(pool_size=2),
        layers.GlobalAveragePooling1D(),
        layers.Dense(dense_units, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Preview parameter counts
for name, cfg in MODEL_CONFIGS.items():
    m = build_har_cnn(cfg["f1"], cfg["f2"], cfg["dense"])
    print(f"{name.upper():6s} | Params: {m.count_params():,}")

SMALL  | Params: 1,398
MEDIUM | Params: 4,582
LARGE  | Params: 16,326


In [ ]:
import pandas as pd

EXPORT_DIR = "/content/exported_models"
os.makedirs(EXPORT_DIR, exist_ok=True)

def representative_dataset():
    # 300 calibration samples from training set for INT8 activation range
    for i in range(300):
        yield [X_train[i:i+1].astype(np.float32)]

def eval_tflite(tflite_path, X, y):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    in_det = interpreter.get_input_details()[0]
    out_det = interpreter.get_output_details()[0]

    correct = 0
    for i in range(len(X)):
        sample = X[i:i+1]
        if in_det["dtype"] == np.int8:
            scale, zero_pt = in_det["quantization"]
            sample = np.round(sample / scale + zero_pt).astype(np.int8)
        else:
            sample = sample.astype(np.float32)

        interpreter.set_tensor(in_det["index"], sample)
        interpreter.invoke()
        out = interpreter.get_tensor(out_det["index"])[0]
        if np.argmax(out) == y[i]:
            correct += 1
    return correct / len(X)

results = []

for name, cfg in MODEL_CONFIGS.items():
    print(f"\n=== Training {name.upper()} ===")
    model = build_har_cnn(cfg["f1"], cfg["f2"], cfg["dense"])
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5, restore_best_weights=True
    )
    model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=30,
        batch_size=64,
        callbacks=[early_stop],
        verbose=1
    )

    # 1. Export FP32 .tflite
    fp32_path = os.path.join(EXPORT_DIR, f"har_{name}_fp32.tflite")
    conv_fp32 = tf.lite.TFLiteConverter.from_keras_model(model)
    with open(fp32_path, "wb") as f:
        f.write(conv_fp32.convert())

    # 2. Export Full-Integer INT8 .tflite (weights + activations + I/O)
    int8_path = os.path.join(EXPORT_DIR, f"har_{name}_int8.tflite")
    conv_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
    conv_int8.optimizations = [tf.lite.Optimize.DEFAULT]
    conv_int8.representative_dataset = representative_dataset
    conv_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv_int8.inference_input_type = tf.int8
    conv_int8.inference_output_type = tf.int8
    with open(int8_path, "wb") as f:
        f.write(conv_int8.convert())

    # 3. Verify accuracy on full UCI HAR test set
    acc_fp32 = eval_tflite(fp32_path, X_test, y_test)
    acc_int8 = eval_tflite(int8_path, X_test, y_test)

    results.append({
        "model": name,
        "params": model.count_params(),
        "fp32_kb": round(os.path.getsize(fp32_path) / 1024, 2),
        "int8_kb": round(os.path.getsize(int8_path) / 1024, 2),
        "fp32_test_acc": round(acc_fp32 * 100, 2),
        "int8_test_acc": round(acc_int8 * 100, 2),
        "acc_drop_pct": round((acc_fp32 - acc_int8) * 100, 2),
    })

df = pd.DataFrame(results)
csv_path = os.path.join(EXPORT_DIR, "pc_baseline_results.csv")
df.to_csv(csv_path, index=False)
print("\n=== SUMMARY (Deliverable D1) ===")
display(df)


=== Training SMALL ===
Epoch 1/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.4481 - loss: 1.5918 - val_accuracy: 0.5150 - val_loss: 1.2634
Epoch 2/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6942 - loss: 0.8386 - val_accuracy: 0.7217 - val_loss: 0.7119
Epoch 3/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8030 - loss: 0.5001 - val_accuracy: 0.7788 - val_loss: 0.5945
Epoch 4/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8558 - loss: 0.3733 - val_accuracy: 0.8250 - val_loss: 0.5343
Epoch 5/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8953 - loss: 0.2920 - val_accuracy: 0.8432 - val_loss: 0.4854
Epoch 6/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9245 - loss: 0.2318 - val_accuracy: 0.8504 - val_loss: 0.4521
Epoch 7/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9378 - loss: 0.1914 - val_accuracy: 0.8558 - val_loss: 0.4324
Epoch 8/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9424 - loss: 0.1655 - val

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



=== Training MEDIUM ===
Epoch 1/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.5737 - loss: 1.2625 - val_accuracy: 0.7289 - val_loss: 0.7440
Epoch 2/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8198 - loss: 0.4918 - val_accuracy: 0.8069 - val_loss: 0.5568
Epoch 3/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8971 - loss: 0.3093 - val_accuracy: 0.8214 - val_loss: 0.4579
Epoch 4/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9337 - loss: 0.2170 - val_accuracy: 0.8622 - val_loss: 0.3813
Epoch 5/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9434 - loss: 0.1734 - val_accuracy: 0.8867 - val_loss: 0.3437
Epoch 6/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9478 - loss: 0.1499 - val_accuracy: 0.8948 - val_loss: 0.3320
Epoch 7/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9506 - loss: 0.1354 - val_accuracy: 0.8966 - val_loss: 0.3266
Epoch 8/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9525 - loss: 0.1259 -

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



=== Training LARGE ===
Epoch 1/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.7060 - loss: 0.8845 - val_accuracy: 0.7833 - val_loss: 0.6276
Epoch 2/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.8587 - loss: 0.3567 - val_accuracy: 0.8178 - val_loss: 0.5159
Epoch 3/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9200 - loss: 0.2313 - val_accuracy: 0.8604 - val_loss: 0.4327
Epoch 4/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9440 - loss: 0.1597 - val_accuracy: 0.8785 - val_loss: 0.3818
Epoch 5/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9507 - loss: 0.1275 - val_accuracy: 0.8803 - val_loss: 0.3645
Epoch 6/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9563 - loss: 0.1101 - val_accuracy: 0.8812 - val_loss: 0.3762
Epoch 7/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9584 - loss: 0.1007 - val_accuracy: 0.8812 - val_loss: 0.3882
Epoch 8/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9613 - loss: 0.0951 - 

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



=== SUMMARY (Deliverable D1) ===


,model,params,fp32_kb,int8_kb,fp32_test_acc,int8_test_acc,acc_drop_pct
0,small,1398,11.53,9.41,89.58,89.82,-0.24
1,medium,4582,23.96,13.57,88.80,88.60,0.20
2,large,16326,69.89,27.18,92.43,92.43,0.00


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
np.random.seed(42)

# 1. Channel-wise Z-score standardisation (fit ONLY on X_train)
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train_norm = ((X_train - mean) / std).astype(np.float32)
X_test_norm  = ((X_test - mean) / std).astype(np.float32)

EXPORT_DIR = "/content/exported_models_v2"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save scaler params for embedded/Pi 5 pre-processing
np.savez(os.path.join(EXPORT_DIR, "scaler_params.npz"), mean=mean.squeeze(), std=std.squeeze())

# 2. Optimised Small, Medium, Large 1D-CNN architectures
MODEL_CONFIGS = {
    "small":  {"f1": 8,  "f2": 16, "dense": 16, "drop": 0.20},
    "medium": {"f1": 16, "f2": 32, "dense": 32, "drop": 0.25},
    "large":  {"f1": 32, "f2": 64, "dense": 64, "drop": 0.30},
}

def build_har_cnn(f1, f2, dense_units, drop_rate, input_shape=(128, 9), num_classes=6):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(f1, kernel_size=5, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(drop_rate),

        layers.Conv1D(f2, kernel_size=5, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(drop_rate),

        layers.GlobalAveragePooling1D(),
        layers.Dense(dense_units, activation="relu"),
        layers.Dropout(drop_rate),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=2e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# 3. Stratified representative dataset (500 random samples across all classes)
rng = np.random.default_rng(42)
cal_indices = rng.choice(len(X_train_norm), size=500, replace=False)

def representative_dataset():
    for i in cal_indices:
        yield [X_train_norm[i:i+1]]

def eval_tflite(tflite_path, X, y):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    in_det = interpreter.get_input_details()[0]
    out_det = interpreter.get_output_details()[0]

    correct = 0
    for i in range(len(X)):
        sample = X[i:i+1]
        if in_det["dtype"] == np.int8:
            scale, zero_pt = in_det["quantization"]
            sample = np.clip(np.round(sample / scale + zero_pt), -128, 127).astype(np.int8)
        else:
            sample = sample.astype(np.float32)

        interpreter.set_tensor(in_det["index"], sample)
        interpreter.invoke()
        out = interpreter.get_tensor(out_det["index"])[0]
        if np.argmax(out) == y[i]:
            correct += 1
    return correct / len(X)

# 4. Train, export, and evaluate
results = []
for name, cfg in MODEL_CONFIGS.items():
    print(f"\n=== Training Optimised {name.upper()} ===")
    model = build_har_cnn(cfg["f1"], cfg["f2"], cfg["dense"], cfg["drop"])

    callbacks = [
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy", factor=0.5, patience=4, min_lr=1e-5, verbose=0
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=12, restore_best_weights=True
        )
    ]

    model.fit(
        X_train_norm, y_train,
        validation_split=0.15,
        epochs=60,
        batch_size=64,
        callbacks=callbacks,
        verbose=1
    )

    # Export FP32
    fp32_path = os.path.join(EXPORT_DIR, f"har_{name}_fp32.tflite")
    conv_fp32 = tf.lite.TFLiteConverter.from_keras_model(model)
    with open(fp32_path, "wb") as f:
        f.write(conv_fp32.convert())

    # Export Full-INT8
    int8_path = os.path.join(EXPORT_DIR, f"har_{name}_int8.tflite")
    conv_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
    conv_int8.optimizations = [tf.lite.Optimize.DEFAULT]
    conv_int8.representative_dataset = representative_dataset
    conv_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv_int8.inference_input_type = tf.int8
    conv_int8.inference_output_type = tf.int8
    with open(int8_path, "wb") as f:
        f.write(conv_int8.convert())

    acc_fp32 = eval_tflite(fp32_path, X_test_norm, y_test)
    acc_int8 = eval_tflite(int8_path, X_test_norm, y_test)

    results.append({
        "model": name,
        "params": model.count_params(),
        "fp32_kb": round(os.path.getsize(fp32_path) / 1024, 2),
        "int8_kb": round(os.path.getsize(int8_path) / 1024, 2),
        "fp32_test_acc": round(acc_fp32 * 100, 2),
        "int8_test_acc": round(acc_int8 * 100, 2),
        "acc_drop_pct": round((acc_fp32 - acc_int8) * 100, 2),
    })

df_v2 = pd.DataFrame(results)
df_v2.to_csv(os.path.join(EXPORT_DIR, "pc_baseline_results_v2.csv"), index=False)
print("\n=== OPTIMISED SUMMARY (Deliverable D1 v2) ===")
display(df_v2)


=== Training Optimised SMALL ===
Epoch 1/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.6327 - loss: 1.0619 - val_accuracy: 0.7860 - val_loss: 0.7170 - learning_rate: 0.0020
Epoch 2/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7793 - loss: 0.4956 - val_accuracy: 0.8767 - val_loss: 0.4401 - learning_rate: 0.0020
Epoch 3/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8373 - loss: 0.3900 - val_accuracy: 0.8966 - val_loss: 0.3606 - learning_rate: 0.0020
Epoch 4/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8718 - loss: 0.3259 - val_accuracy: 0.9102 - val_loss: 0.2864 - learning_rate: 0.0020
Epoch 5/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8981 - loss: 0.2702 - val_accuracy: 0.8976 - val_loss: 0.2764 - learning_rate: 0.0020
Epoch 6/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9091 - loss: 0.2368 - val_accuracy: 0.9193 - val_loss: 0.2468 - learning_rate: 0.0020
Epoch 7/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accurac

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



=== Training Optimised MEDIUM ===
Epoch 1/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.6819 - loss: 0.7942 - val_accuracy: 0.7480 - val_loss: 0.5058 - learning_rate: 0.0020
Epoch 2/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9102 - loss: 0.2667 - val_accuracy: 0.8976 - val_loss: 0.2871 - learning_rate: 0.0020
Epoch 3/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9387 - loss: 0.1764 - val_accuracy: 0.9012 - val_loss: 0.2691 - learning_rate: 0.0020
Epoch 4/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9466 - loss: 0.1479 - val_accuracy: 0.8994 - val_loss: 0.2752 - learning_rate: 0.0020
Epoch 5/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9474 - loss: 0.1338 - val_accuracy: 0.9012 - val_loss: 0.2773 - learning_rate: 0.0020
Epoch 6/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9515 - loss: 0.1243 - val_accuracy: 0.9084 - val_loss: 0.2829 - learning_rate: 0.0020
Epoch 7/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accura

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



=== Training Optimised LARGE ===
Epoch 1/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.8216 - loss: 0.4875 - val_accuracy: 0.9102 - val_loss: 0.2717 - learning_rate: 0.0020
Epoch 2/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.9394 - loss: 0.1683 - val_accuracy: 0.9039 - val_loss: 0.2607 - learning_rate: 0.0020
Epoch 3/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9480 - loss: 0.1385 - val_accuracy: 0.9030 - val_loss: 0.2804 - learning_rate: 0.0020
Epoch 4/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.9522 - loss: 0.1217 - val_accuracy: 0.9139 - val_loss: 0.2947 - learning_rate: 0.0020
Epoch 5/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9554 - loss: 0.1081 - val_accuracy: 0.9057 - val_loss: 0.2929 - learning_rate: 0.0020
Epoch 6/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9552 - loss: 0.1083 - val_accuracy: 0.9030 - val_loss: 0.3170 - learning_rate: 0.0020
Epoch 7/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accurac

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



=== OPTIMISED SUMMARY (Deliverable D1 v2) ===


,model,params,fp32_kb,int8_kb,fp32_test_acc,int8_test_acc,acc_drop_pct
0,small,1470,12.72,10.59,91.79,91.45,0.34
1,medium,4726,25.20,14.79,93.35,93.35,0.00
2,large,16614,71.07,28.37,92.40,91.96,0.44


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/har_models_d1_v2", "zip", "/content/exported_models_v2")
files.download("/content/har_models_d1_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>